In [1]:
from google.cloud import vision_v1
from google.cloud import vision
from gtts import gTTS
import os
import cv2
import numpy as np
import re

In [2]:
from src.ocr.vision_ocr import TextExtractor
from src.llm_model.langchain import LangChainSetup

In [3]:
image_path = "/Users/rasikagulhane/Desktop/ReadingOut-Prescription-label/dataset/Prescription_1.png"

In [4]:

def detect_document(path):
    """Detects document features in an image."""
    # Use the service account file for authentication
    client = vision.ImageAnnotatorClient.from_service_account_file(os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

    with open(path, "rb") as image_file:
        content = image_file.read()
    image = vision.Image(content=content)
    response = client.document_text_detection(image=image)

    for page in response.full_text_annotation.pages:
        for block in page.blocks:
            print(f"\nBlock confidence: {block.confidence}\n")
            for paragraph in block.paragraphs:
                print("Paragraph confidence: {}".format(paragraph.confidence))
                for word in paragraph.words:
                    word_text = "".join([symbol.text for symbol in word.symbols])
                    print(
                        "Word text: {} (confidence: {})".format(
                            word_text, word.confidence
                        )
                    )
                    for symbol in word.symbols:
                        print(
                            "\tSymbol: {} (confidence: {})".format(
                                symbol.text, symbol.confidence
                            )
                        )
    if response.error.message:
        raise Exception(
            "{}\nFor more info on error messages, check: "
            "https://cloud.google.com/apis/design/errors".format(response.error.message)
        )

detect_document(image_path)


Block confidence: 0.9853841066360474

Paragraph confidence: 0.9853841066360474
Word text: DEA (confidence: 0.9841700792312622)
	Symbol: D (confidence: 0.9859437942504883)
	Symbol: E (confidence: 0.9833112359046936)
	Symbol: A (confidence: 0.9832552671432495)
Word text: # (confidence: 0.9490182995796204)
	Symbol: # (confidence: 0.9490182995796204)
Word text: GB (confidence: 0.9902670979499817)
	Symbol: G (confidence: 0.9890341758728027)
	Symbol: B (confidence: 0.9915000200271606)
Word text: 05455616 (confidence: 0.9891643524169922)
	Symbol: 0 (confidence: 0.9930540323257446)
	Symbol: 5 (confidence: 0.993011474609375)
	Symbol: 4 (confidence: 0.9914464354515076)
	Symbol: 5 (confidence: 0.9902170896530151)
	Symbol: 5 (confidence: 0.989607036113739)
	Symbol: 6 (confidence: 0.9842172861099243)
	Symbol: 1 (confidence: 0.987696647644043)
	Symbol: 6 (confidence: 0.9840648770332336)

Block confidence: 0.9560270309448242

Paragraph confidence: 0.9560270309448242
Word text: dobe (confidence: 0.95

In [5]:

text_extractor = TextExtractor(image_path)
text_data = text_extractor.extract_text()
cleaned_data = TextExtractor.clean_text(text_data)

cleaned_data

['DEA GB 05455616\ndobe\nAdobe Stock\nMEDICAL CENTRE\n824 14th Street\nNew York NY 91743 USA\nNAME John Smith\naddress 162 Example St NY\nR\nAdobe Ste\nBetaloc 100 ng\nAdobe\n100tab BID\nDorzolamidum 10\nCimetidine 50\n\n-1\n-\nmg -\n-1 tab BID\n2 tabs TID\nOxprelol 50mg - 1 tab\nAdo\nQD\n LABEL\nREFILL 012 3 4 5 PRN\nLIC  976269\nAGE\n34\nDATE 09-11-12\nStock\nAdobe Sto\nDr Steve Johnson\nsignature\nAdobe Sock\nWTX-N-PRESC-T\n1-889-422-0700']

In [6]:
# Medical information Clean Text:

for cleaned_text in cleaned_data:
    print(cleaned_text)

lang_chain_setup = LangChainSetup()
response_created = lang_chain_setup.setup_chain(cleaned_text)
# print(response_created)


DEA GB 05455616
dobe
Adobe Stock
MEDICAL CENTRE
824 14th Street
New York NY 91743 USA
NAME John Smith
address 162 Example St NY
R
Adobe Ste
Betaloc 100 ng
Adobe
100tab BID
Dorzolamidum 10
Cimetidine 50

-1
-
mg -
-1 tab BID
2 tabs TID
Oxprelol 50mg - 1 tab
Ado
QD
 LABEL
REFILL 012 3 4 5 PRN
LIC  976269
AGE
34
DATE 09-11-12
Stock
Adobe Sto
Dr Steve Johnson
signature
Adobe Sock
WTX-N-PRESC-T
1-889-422-0700


/Users/rasikagulhane/Desktop/ReadingOut-Prescription-label/renv/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
/Users/rasikagulhane/Desktop/ReadingOut-Prescription-label/renv/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `__call__` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

        Text:DEA GB 05455616
dobe
Adobe Stock
MEDICAL CENTRE
824 14th Street
New York NY 91743 USA
NAME John Smith
address 162 Example St NY
R
Adobe Ste
Betaloc 100 ng
Adobe
100tab BID
Dorzolamidum 10
Cimetidine 50

-1
-
mg -
-1 tab BID
2 tabs TID
Oxprelol 50mg - 1 tab
Ado
QD
 LABEL
REFILL 012 3 4 5 PRN
LIC  976269
AGE
34
DATE 09-11-12
Stock
Adobe Sto
Dr Steve Johnson
signature
Adobe Sock
WTX-N-PRESC-T
1-889-422-0700
        You are an expert of Medicines. Given the above text, it is your job to extract the medicine information from it.
        Make sure you extract all medicine from text, check all medicines Dosage value in mg and Dosage frequency carefully.
        Make sure to format your response like RESPONSE_JSON below and use it as a guide.         Ensure to extract all medicine from text.
        ### RESPONSE_JSON
        {'1': {'name': 'name of the Medicine', 'Content Value':

In [7]:

import re

# Use regular expression to extract content after "### RESPONSE_JSON"
match = re.search(r'### RESPONSE_JSON\s*(.*)', response_created, re.DOTALL)
if match:
    input_text = match.group(1).strip()
    print(input_text)
else:
    print("Pattern not found.")


{
    '1': {'name': 'Betaloc', 'Content Value': '100 mg', 'Dosage': '1 tab BID'},
    '2': {'name': 'Dorzolamidum', 'Content Value': '10 mg', 'Dosage': '2 tabs TID'},
    '3': {'name': 'Cimetidine', 'Content Value': '50 mg', 'Dosage': '1 tab BID'},
    '4': {'name': 'Oxprelol', 'Content Value': '50 mg', 'Dosage': '1 tab QD'}
}


### Needfull detection:

In [12]:
detected_text = cleaned_data[0]

In [13]:
#PATIENT FIRST NAME

import re

def extract_first_name(text):
    # Define regular expressions for "NAME" or "Name" followed by a space and the first name
    name_pattern = re.compile(r'\b(?:NAME|Name|Name\n)\s+(\w+)\b', re.IGNORECASE)

    # Search for the pattern in the text
    match = re.search(name_pattern, text)

    # If a match is found, return the first name
    if match:
        return match.group(1)
    else:
        return None

# Example usage
# input_text = "Patient NAME John Doe is scheduled for a checkup."
first_name = extract_first_name(detected_text)

if first_name:
    print("First Name:", first_name)
else:
    print("No first name found.")


First Name: John


In [14]:
#PHYSICIAN NAME

import re

def detect_and_extract_physician_name(text):
    # Define a regular expression pattern to capture "Dr." or "DR" followed by a name
    pattern = re.compile(r'\b(?:Dr\.|DR|MD)\s+([A-Za-z]+\s*[A-Za-z]*)\b', re.IGNORECASE)

    # Find the first match in the text
    match = re.search(pattern, text)

    # If a match is found, extract the physician's name
    if match:
        physician_name = match.group(1)
        return True, physician_name.strip()  # Trim leading/trailing spaces
    else:
        return False, None



is_physician_name_detected, physician_name = detect_and_extract_physician_name(detected_text)

if is_physician_name_detected:
    print("Physician's Name Detected: Dr.", physician_name)
else:
    print("No physician's name detected.")


Physician's Name Detected: Dr. Steve Johnson


In [15]:
#DATE FILTER

import re
from datetime import datetime

def detect_date(text):
    # Define common date patterns using regular expressions
    date_patterns = [
        r'\b(\d{1,2}/\d{1,2}/\d{2,4})\b',     # MM/DD/YYYY or MM/DD/YY
        r'\b(\d{4}-\d{1,2}-\d{1,2})\b',        # YYYY-MM-DD
        r'\b(\d{1,2}-[A-Za-z]{3}-\d{2,4})\b',  # DD-Mon-YYYY or DD-Mon-YY
        r'\bDATE (\d{2}-\d{2}-\d{2})\b',
        r'Date (\d{1,2}-\d{1,2}-\d{4})',
        # r'Date\n(.+)',
        r'(\d{2}-\d{2}-\d{4})',
        r'(\d{2}-\d{2}-\d{4}|\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}|\d{4}/\d{2}/\d{2})'

 
    ]

    # Iterate through date patterns and find matches
    for pattern in date_patterns:
        match = re.search(pattern, text)
        if match:
            date = match.group(1)
            # Convert the detected date to a standardized format (optional)
            standardized_date = convert_to_standard_format(date)
            return True, standardized_date

    return False, None

def convert_to_standard_format(date_str):
    # Convert detected date to a standardized format (optional)
    try:
        parsed_date = datetime.strptime(date_str, '%m/%d/%Y')
        return parsed_date.strftime('%Y-%m-%d')
    except ValueError:
        try:
            parsed_date = datetime.strptime(date_str, '%Y-%m-%d')
            return parsed_date.strftime('%Y-%m-%d')
        except ValueError:
            return date_str

is_date_detected, date = detect_date(detected_text)

if is_date_detected:
    print("Date Detected:", date)
else:
    print("No date detected.")


Date Detected: 09-11-12


In [16]:
# MEDICINE DETAILS

# input_text = response_created  # (llm output)

# input_text = '''
# {
#     '1': {'name': 'CALPOL', 'Content Value': '250 mg', 'Dosage': '4 ml every 6 hours for 3 days'},
#     '2': {'name': 'DELCON', 'Content Value': '3 ml', 'Dosage': 'Take 9 ml daily for 5 days'},
#     '3': {'name': 'LEVOLIN', 'Content Value': '3 ml', 'Dosage': 'Take once daily for 5 days'},
#     '4': {'name': 'MEFTAL-P', 'Content Value': '100 mg', 'Dosage': 'Take 3 ml as needed'}
# }
# '''


import re

def clean_and_replace(input_text):
    # Define a function to replace initial numbers with ordinal numbers
    def replace_numbers(match):
        number = match.group(1)
        return '{}{}'.format(number, ordinal_suffix(int(number)))

    # Replace initial numbers with ordinal numbers
    cleaned_text = re.sub(r'\'(\d+)\':', replace_numbers, input_text)

    # Remove specific characters and clean the text
    cleaned_text = re.sub(r"[{}',]", '', cleaned_text)

    # Replace dosage abbreviations with their meanings
    cleaned_text = re.sub(r'\b(Tab|BID|TID|TTD|QD|PO)\b', lambda match: map_dosage_frequency(match.group(1)), cleaned_text)

    return cleaned_text

def map_dosage_frequency(dosage_abbreviation):
    # Define a dictionary to map dosage abbreviations to their meanings
    dosage_mapping = {
        'tab': 'Tablet'
        'BID': 'twice a day',
        '1-0-1': 'twice a day',
        '1-1': 'twice a day',
        'TID': 'three times a day',
        'TTD': 'three times a day',
        '1-1-1': 'three times a day',
        'QD': 'once a day',
        'OD': 'Once a day',
        'PO': 'By mouth'
        
        # Add more mappings as needed
    }
    return dosage_mapping.get(dosage_abbreviation, dosage_abbreviation)

def ordinal_suffix(number):
    if 10 <= number % 100 <= 20:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(number % 10, 'th')
    return suffix


Medicine_info = clean_and_replace(input_text)

print("Cleaned Text:")
print(Medicine_info)


Cleaned Text:

    1st name: Betaloc Content Value: 100 mg Dosage: 1 tab twice a day
    2nd name: Dorzolamidum Content Value: 10 mg Dosage: 2 tabs three times a day
    3rd name: Cimetidine Content Value: 50 mg Dosage: 1 tab twice a day
    4th name: Oxprelol Content Value: 50 mg Dosage: 1 tab once a day



In [17]:
#CONTACT INFO 


import re

def detect_contact_info(text):
    contact_info = {
        'Phone Numbers': [],
        'Email Addresses': []
    }

    # Detect phone numbers
    phone_number_pattern = re.compile(r'\b(?:\+?[0-9]{1,3}[-. ]?)?\(?[0-9]{3}\)?[-. ]?[0-9]{3}[-. ]?[0-9]{4}\b')
    phone_numbers = re.findall(phone_number_pattern, text)
    contact_info['Phone Numbers'] = phone_numbers

    # Detect email addresses
    email_pattern = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
    emails = re.findall(email_pattern, text)
    contact_info['Email Addresses'] = emails


    if not contact_info['Email Addresses']:
        return contact_info['Phone Numbers']
    elif not contact_info['Phone Numbers']:
        return contact_info['Email Addresses']
    else:
        return contact_info

contact_info = detect_contact_info(detected_text)

print(contact_info)



# # Print detected contact information
# print("Detected Contact Information:")
# print("Phone Numbers:", contact_info['Phone Numbers'])
# print("Email Addresses:", contact_info['Email Addresses'])



['1-889-422-0700']


### TTS

In [ ]:
# ## TTS part

# from pygame import mixer
# import tempfile

# def text_to_speech(text):
#     # Create a gTTS object
#     tts = gTTS(text=text, lang='en', slow=False)

#     # Save the speech as an MP3 file
#     with tempfile.NamedTemporaryFile(delete=False) as temp_mp3:
#         tts.save(temp_mp3.name)
#         mp3_path = temp_mp3.name

#     # Play the MP3 file
#     mixer.init()
#     mixer.music.load(mp3_path)
#     mixer.music.play()


In [ ]:

# # Convert the text to speech
# text_to_speech(detected_text)


In [ ]:
# from gtts import gTTS
# import os

# def text_to_speech(input_file, output_file):
#     # Read text from the file
#     with open(input_file, 'r') as file:
#         text = file.read()

#     # Create a gTTS object
#     tts = gTTS(text, lang='en')  # You can specify the language using the 'lang' parameter

#     # Save the generated speech to an audio file
#     tts.save(output_file)

#     # Play the generated speech using the default audio player
#     os.system(f'start {output_file}')  # This works on Windows, adjust for other operating systems


# if __name__ == "__main__":
#     input_file_path = detected_text # Replace with your input file path
#     output_file_path = 'output.mp3'  # Replace with your desired output file path

#     text_to_speech(input_file_path, output_file_path)


   

In [ ]:
# text_to_speech('output.txt')

### TTS With template:

In [18]:
from google.cloud import vision_v1
import re
from gtts import gTTS
from io import BytesIO
from pygame import mixer
import tempfile

pygame 2.5.2 (SDL 2.28.3, Python 3.11.8)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [19]:
first_name= first_name
physician_name= physician_name
date= date
medicine_info= Medicine_info
contact_info= contact_info

In [22]:
from gtts import gTTS
import os



def create_tts_message(first_name, physician_name, date, medicine_info, contact_info):
    
    if not input_text:
        tts_message = "Hi There! Sorry the given image is not supportive or The written text is very unclear"
        
    # Implement the logic to create the TTS message based on input parameters
    # You can customize this function according to your requirements
    tts_message = f"Hi {first_name},\nThis is a prescription from Doctor {physician_name} on {date}.\nPrescribed medicine are as follows: {medicine_info}"

    # Add "For more information" sentence if contact_info is available
    if contact_info:
        tts_message += f"\nFor more information, you can contact:\n + {contact_info}"

    return tts_message

def text_to_speech(tts_message):
    print("Text to be spoken:")
    print(tts_message)
    
    # Create gTTS object
    tts = gTTS(text=tts_message, lang='en')

    # Save the generated audio file
    tts.save('TTS_Audio/prescription_message4.mp3')

    # Play the audio file
    os.system('start prescription_message4.mp3')


In [23]:


# Create TTS message
tts_message = create_tts_message(first_name, physician_name, date, medicine_info, contact_info)

# Convert the text to speech
text_to_speech(tts_message)


Text to be spoken:
Hi John,
This is a prescription from Doctor Steve Johnson on 09-11-12.
Prescribed medicine are as follows: 
    1st name: Betaloc Content Value: 100 mg Dosage: 1 tab twice a day
    2nd name: Dorzolamidum Content Value: 10 mg Dosage: 2 tabs three times a day
    3rd name: Cimetidine Content Value: 50 mg Dosage: 1 tab twice a day
    4th name: Oxprelol Content Value: 50 mg Dosage: 1 tab once a day

For more information, you can contact:
 + ['1-889-422-0700']


sh: start: command not found


### Rough

In [ ]:
def text_to_speech(text):
    # Create a gTTS object
    tts = gTTS(text=text, lang='en', slow=False)

    # Save the speech as an MP3 file
    with tempfile.NamedTemporaryFile(delete=False) as temp_mp3:
        tts.save(temp_mp3.name)
        mp3_path = temp_mp3.name

    # Play the MP3 file
    mixer.init()
    mixer.music.load(mp3_path)
    mixer.music.play()
